In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# --- Part 1: Create a dummy CSV for testing (since the file is missing) ---
data_dict = {'temperature': np.random.randint(20, 40, size=100)}
df = pd.DataFrame(data_dict)
df.to_csv('temperature.csv', index=False)

# --- Part 2: Extracted Experiment Code ---
data = pd.read_csv('temperature.csv')
print("Columns:", data.columns)

# Select the first numerical column automatically
temp_col = data.select_dtypes(include=[np.number]).columns[0]
print("Using Column:", temp_col)

# Clean and reshape data
temp_data = data[[temp_col]].dropna().values
temp_data = temp_data.reshape(-1, 1)

# Normalize data
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(temp_data)

x, y = [], []
# Create sequences using 30 past days to predict the next day
for i in range(30, len(data_scaled)):
    x.append(data_scaled[i-30:i, 0])
    y.append(data_scaled[i, 0])

x, y = np.array(x), np.array(y)

# Reshape for LSTM: [samples, time_steps, features]
x = x.reshape((x.shape[0], x.shape[1], 1))

# Build LSTM model
model = Sequential([
    LSTM(50, activation='relu', input_shape=(x.shape[1], 1)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(x, y, epochs=20, batch_size=16, verbose=0)

# Prediction
pred = model.predict(x)
# Invert scaling to get actual temperature values
pred = scaler.inverse_transform(pred)

print("Predictions (first 5):")
print(pred[:5])

Columns: Index(['temperature'], dtype='object')
Using Column: temperature


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Predictions (first 5):
[[28.429676]
 [28.851248]
 [28.94059 ]
 [28.853727]
 [28.469446]]
